In [ ]:
# ============================================================
# LIGHTGBM — FINAL COMPACT KAGGLE CODE
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from tqdm.auto import tqdm


# ============================
# 1. Load Data
# ============================

KAGGLE_DATASET_URL = "/kaggle/input/competitions/playground-series-s6e9/"

def find_data_dir() -> Path:
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd / "notebook",
        cwd.parent,
        cwd.parent / "notebook",
    ])

    for base in candidates:
        for candidate in [base / "input", base.parent / "input"]:
            if candidate.exists() and (candidate / "train.csv").exists() and (candidate / "test.csv").exists():
                return candidate

    for kaggle_path in [
        Path(KAGGLE_DATASET_URL),
        Path("/kaggle/input/train.csv").parent,
    ]:
        if kaggle_path.exists() and (kaggle_path / "train.csv").exists() and (kaggle_path / "test.csv").exists():
            return kaggle_path

    return Path(KAGGLE_DATASET_URL)


data_dir = find_data_dir()

train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")
sample_submission = pd.read_csv(data_dir / "sample_submission.csv")


# ============================
# 2. Prepare Data
# ============================

id_column = sample_submission.columns[0]
target_column = sample_submission.columns[1]

X = train.drop(columns=[id_column, target_column]).copy()
X_test = test.drop(columns=[id_column]).copy()

y = train[target_column].map({
    "No": 0,
    "Yes": 1
}).astype("int8")

categorical_columns = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in categorical_columns:
    categories = pd.concat(
        [X[col], X_test[col]]
    ).astype("category").cat.categories

    X[col] = pd.Categorical(
        X[col],
        categories=categories
    )

    X_test[col] = pd.Categorical(
        X_test[col],
        categories=categories
    )


# ============================
# 3. LightGBM Parameters
# ============================

params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.03,
    "n_estimators": 3000,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 30,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1
}


# ============================
# 4. 5-Fold OOF
# ============================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof = np.zeros(len(X))
fold_auc = []
best_iterations = []

for fold, (tr, va) in tqdm(
    enumerate(cv.split(X, y), 1),
    total=5,
    desc="LightGBM",
    unit="fold"
):

    model = lgb.LGBMClassifier(**params)

    model.fit(
        X.iloc[tr],
        y.iloc[tr],
        eval_set=[(X.iloc[va], y.iloc[va])],
        eval_metric="auc",
        categorical_feature=categorical_columns,
        callbacks=[
            lgb.early_stopping(
                100,
                verbose=False
            )
        ]
    )

    pred = model.predict_proba(
        X.iloc[va]
    )[:, 1]

    oof[va] = pred

    score = roc_auc_score(
        y.iloc[va],
        pred
    )

    fold_auc.append(score)
    best_iterations.append(model.best_iteration_)

    print(
        f"Fold {fold}: "
        f"AUC={score:.8f} | "
        f"iter={model.best_iteration_}"
    )


# ============================
# 5. OOF Results
# ============================

oof_auc = roc_auc_score(y, oof)
baseline = 0.941407813895485

mean_iteration = max(
    1,
    int(round(np.mean(best_iterations)))
)


print("\n" + "=" * 60)
print("OOF RESULTS")
print("=" * 60)

print(
    f"Fold AUCs      : "
    f"{[round(x, 6) for x in fold_auc]}"
)

print(
    f"Mean Fold AUC  : "
    f"{np.mean(fold_auc):.6f}"
)

print(
    f"Fold Std       : "
    f"{np.std(fold_auc):.6f}"
)

print(
    f"OOF AUC        : "
    f"{oof_auc:.15f}"
)

print(
    f"Previous HGB   : "
    f"{baseline:.15f}"
)

print(
    f"Improvement    : "
    f"{oof_auc - baseline:+.15f}"
)

print(
    f"Best Iterations: "
    f"{best_iterations}"
)

print(
    f"Mean Iteration : "
    f"{mean_iteration}"
)


# ============================
# 6. Final Model
# ============================

params["n_estimators"] = mean_iteration

final_model = lgb.LGBMClassifier(**params)

final_model.fit(
    X,
    y,
    categorical_feature=categorical_columns
)


# ============================
# 7. Diagnostics
# ============================

train_pred = final_model.predict_proba(X)[:, 1]

train_auc = roc_auc_score(
    y,
    train_pred
)

test_pred = final_model.predict_proba(
    X_test
)[:, 1]


# ============================
# 8. Submission
# ============================

submission = sample_submission.copy()

submission[target_column] = test_pred

output_path = "submission.parquet"

submission.to_parquet(output_path, index=False)


# ============================
# 9. Final Output
# ============================

print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

print(f"Training AUC : {train_auc:.15f}")
print(f"OOF AUC      : {oof_auc:.15f}")
print(f"AUC Gap      : {train_auc - oof_auc:+.15f}")
print(f"Improvement  : {oof_auc - baseline:+.15f}")
print(f"Test rows    : {len(test_pred):,}")
print(f"Submission   : {output_path}")

print("\nPrediction statistics:")
print(pd.Series(test_pred).describe())

print("\nSubmission preview:")
print(submission.head())


LightGBM:   0%|          | 0/5 [00:00<?, ?fold/s]

Fold 1: AUC=0.94076842 | iter=1463
Fold 2: AUC=0.94163583 | iter=1086
Fold 3: AUC=0.94299380 | iter=960
Fold 4: AUC=0.94245597 | iter=958
Fold 5: AUC=0.94186140 | iter=1023

OOF RESULTS
Fold AUCs      : [0.940768, 0.941636, 0.942994, 0.942456, 0.941861]
Mean Fold AUC  : 0.941943
Fold Std       : 0.000755
OOF AUC        : 0.941931571271773
Previous HGB   : 0.941407813895485
Improvement    : +0.000523757376288
Best Iterations: [1463, 1086, 960, 958, 1023]
Mean Iteration : 1098

FINAL SUMMARY
Training AUC : 0.945509780253442
OOF AUC      : 0.941931571271773
AUC Gap      : +0.003578208981669
Improvement  : +0.000523757376288
Test rows    : 286,571
Submission   : submission_lightgbm.parquet

Prediction statistics:
count    286571.000000
mean          0.174638
std           0.269977
min           0.000024
25%           0.002705
50%           0.018002
75%           0.267641
max           0.966349
dtype: float64

Submission preview:
       id  Will_Buy_EV
0  668665     0.009147
1  668666     0